prueba commit 2

# Verificación del corpus limpio
La extracción y limpieza principal se hace ahora dentro de `extraccion_corpus_mixto.py`.

Este notebook sirve para comprobar el CSV final y la trazabilidad de filtros después del reprocesado.

In [ ]:
import pandas as pd

FINAL_CSV = 'data/corpus/master_corpus_mixto_clean.csv'
TRACE_CSV = 'data/corpus/master_corpus_mixto_traceability.csv'

final_df = pd.read_csv(FINAL_CSV)
trace_df = pd.read_csv(TRACE_CSV)

print('Final CSV rows:', len(final_df))
print('Traceability rows:', len(trace_df))
print('\nFinal columns:')
print(', '.join(final_df.columns))
print('\nFilter status counts:')
print(trace_df['filter_status'].value_counts(dropna=False).to_string())
print('\nFilter reasons sample:')
print(trace_df['filter_reason'].fillna('').value_counts().head(10).to_string())
print('\nPreview of final CSV:')
print(final_df[['doc_id', 'title', 'year', 'doi', 'language', 'keywords']].head(5).to_string(index=False, max_colwidth=120))

Final CSV rows: 700
Traceability rows: 1000

Final columns:
doc_id, title, abstract, clean_abstract, year, doi, source, authors, keywords, journal, language, top_terms_no_stopwords

Filter status counts:
filter_status
kept       700
dropped    300

Filter reasons sample:
filter_reason
                                                       700
abstract_too_short<500                                 269
abstract_empty|language_unknown                          6
abstract_too_short<500|language_not_english:fr           3
duplicate_doi                                            3
language_low_confidence:0.55                             2
abstract_too_short<500|language_not_english:de           2
abstract_too_short<500|language_low_confidence:0.71      2
language_low_confidence:0.57                             1
abstract_too_short<500|language_low_confidence:0.55      1

Preview of final CSV:
      doc_id                                                                                         

In [ ]:
# Celda 3: Preparación para limpieza y EDA
# Qué hace: carga datos, define rutas y funciones de limpieza reutilizables.
# Qué devuelve: tamaños iniciales de los dos CSV y una vista rápida de columnas.

import re
import unicodedata
from collections import Counter
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd

BASE_DIR = Path('.')
FINAL_CSV = BASE_DIR / 'data/corpus/master_corpus_mixto_clean.csv'
TRACE_CSV = BASE_DIR / 'data/corpus/master_corpus_mixto_traceability.csv'
EDA_DIR = BASE_DIR / 'docs' / 'eda'
EDA_DIR.mkdir(parents=True, exist_ok=True)

final_df = pd.read_csv(FINAL_CSV)
trace_df = pd.read_csv(TRACE_CSV)

EDITORIAL_PATTERNS = [
    r'(?i)published by[^.]*\.?',
    r'(?i)correspondence to[^.]*\.?',
    r'(?i)copyright[^.]*\.?',
    r'(?i)received:\s*[^.]*\.?',
    r'(?i)accepted:\s*[^.]*\.?',
    r'(?i)specialty section:\s*[^.]*\.?',
    r'(?i)citation:\s*[^.]*\.?',
    r'(?i)doi:\s*10\.[^\s]+',
    r'http[s]?://\S+',
]

# Stopwords ampliadas para evitar términos funcionales/estructurales en top palabras.
STOPWORDS = {
    'the', 'and', 'for', 'with', 'this', 'that', 'from', 'were', 'are', 'was', 'have', 'has', 'had',
    'into', 'their', 'there', 'than', 'then', 'when', 'where', 'which', 'while', 'within', 'without',
    'using', 'used', 'use', 'also', 'can', 'may', 'might', 'more', 'most', 'some', 'such', 'over',
    'between', 'through', 'about', 'across', 'during', 'results', 'result', 'study', 'paper', 'article',
    'data', 'method', 'methods', 'model', 'models', 'analysis', 'based', 'show', 'shows', 'shown',
    'high', 'low', 'new', 'one', 'two', 'three', 'four', 'five', 'first', 'second', 'third', 'both',
    'these', 'those', 'each', 'other', 'onto', 'per', 'via', 'its', 'our', 'your', 'they', 'them',
    'you', 'we', 'his', 'her', 'not', 'but', 'all', 'any', 'due', 'doi', 'http', 'https', 'www',
    'com', 'org', 'edu', 'introduction', 'abstract', 'keywords', 'index', 'terms',
    'del', 'las', 'los', 'una', 'uno', 'unos', 'unas', 'para', 'por', 'con', 'sin', 'sobre', 'entre',
    'desde', 'hasta', 'como', 'tambien', 'este', 'esta', 'estos', 'estas', 'ese', 'esa', 'esos',
    'esas', 'fue', 'fueron', 'ser', 'son', 'han', 'hace', 'hacer', 'hacia', 'segun',
    'datos', 'estudio', 'resultados', 'metodo', 'metodos', 'analisis', 'introduccion', 'resumen',
    'palabras', 'clave',
}


def fold_token(token: str) -> str:
    value = unicodedata.normalize('NFKD', token)
    return ''.join(ch for ch in value if not unicodedata.combining(ch))


def normalize_text(text: str) -> str:
    if pd.isna(text):
        return ''
    value = str(text)
    value = unicodedata.normalize('NFKC', value)
    value = value.replace('\u00ad', ' ')
    value = value.replace('ﬁ', 'fi').replace('ﬂ', 'fl')
    value = value.replace('\r', ' ').replace('\n', ' ')
    for pattern in EDITORIAL_PATTERNS:
        value = re.sub(pattern, ' ', value)
    value = re.sub(r'\s+', ' ', value)
    value = re.sub(r'\s*([,;:.!?])\s*', r'\1 ', value)
    return value.strip(' .;:-')


def lex_clean(text: str) -> str:
    if not text:
        return ''
    tokens = re.findall(r"[a-zA-ZÀ-ÿ]{3,}", text.lower())
    tokens = [fold_token(t) for t in tokens]
    tokens = [t for t in tokens if t not in STOPWORDS]
    return ' '.join(tokens)


print('Filas en corpus final:', len(final_df))
print('Filas en trazabilidad:', len(trace_df))
print('\nColumnas del corpus final:')
print(final_df.columns.tolist())

## Tarea 4.1 - Limpieza textual (abstract_raw y abstract_norm)

En esta sección creamos columnas de limpieza para que el input sea útil para embeddings/BERT.

- `abstract_raw`: copia del abstract extraído.
- `abstract_norm`: abstract normalizado (espacios/saltos/caracteres raros/texto editorial).
- `clean_abstract_lex`: versión léxica para análisis de términos (sin stopwords).

La celda siguiente devuelve ejemplos comparando antes/después de limpieza.

In [ ]:
# Celda 5: Construcción del dataset limpio para NLP
# Qué hace: crea abstract_raw, abstract_norm y clean_abstract_lex.
# Qué devuelve: muestra comparativa de 5 filas y estadísticas de longitud.

eda_df = final_df.copy()
eda_df['abstract_raw'] = eda_df['abstract'].fillna('')
eda_df['abstract_norm'] = eda_df['abstract_raw'].map(normalize_text)
eda_df['clean_abstract_lex'] = eda_df['abstract_norm'].map(lex_clean)
eda_df['abstract_norm_len'] = eda_df['abstract_norm'].str.len()

cols_show = ['doc_id', 'abstract_raw', 'abstract_norm', 'clean_abstract_lex']
preview = eda_df[cols_show].head(5).copy()
pd.set_option('display.max_colwidth', 130)
print('Vista de limpieza (primeras 5 filas):')
print(preview.to_string(index=False))

print('\nResumen longitud abstract_norm (caracteres):')
print(eda_df['abstract_norm_len'].describe(percentiles=[0.1, 0.25, 0.5, 0.75, 0.9]).to_string())

## Tarea 5 - EDA del corpus final

En esta parte analizamos tamaño, cobertura temporal, longitudes, calidad de campos, términos frecuentes y sesgos/huecos.

La siguiente celda genera:

- métricas de tamaño y cobertura,
- tabla de nulos,
- distribución por año y por fuente,
- top unigramas y bigramas,
- razones de descarte desde trazabilidad.

In [ ]:
# Celda 7: Métricas EDA (tablas)
# Qué hace: calcula cobertura, nulos, distribución temporal y top términos.
# Qué devuelve: tablas listas para interpretar o exportar al informe.

# 1) Tamaño y cobertura
n_processed = len(trace_df)
n_with_abstract = int((trace_df['abstract_length'].fillna(0) > 0).sum())
n_final = len(eda_df)

coverage = pd.DataFrame([
    {'metrica': 'documentos_procesados', 'valor': n_processed},
    {'metrica': 'con_abstract_no_vacio_trazabilidad', 'valor': n_with_abstract},
    {'metrica': 'final_tras_filtros', 'valor': n_final},
])
print('=== Tamaño y cobertura ===')
print(coverage.to_string(index=False))

# 2) Distribución por año
year_dist = (
    pd.to_numeric(eda_df['year'], errors='coerce')
    .dropna()
    .astype(int)
    .value_counts()
    .sort_index()
    .rename_axis('year')
    .reset_index(name='n')
)
print('\n=== Distribución por año (primeras filas) ===')
print(year_dist.head(15).to_string(index=False))

# 3) Nulos de campos clave
null_table = pd.DataFrame({
    'campo': ['year', 'doi', 'journal', 'keywords', 'authors', 'abstract_norm'],
    'null_pct': [
        eda_df['year'].isna().mean() * 100,
        eda_df['doi'].isna().mean() * 100,
        eda_df['journal'].isna().mean() * 100,
        eda_df['keywords'].isna().mean() * 100,
        eda_df['authors'].isna().mean() * 100,
        (eda_df['abstract_norm'].str.len() == 0).mean() * 100,
    ],
})
print('\n=== Tabla de nulos (%) ===')
print(null_table.to_string(index=False, float_format=lambda x: f'{x:.2f}'))

# 4) Fuentes
source_dist = eda_df['source'].fillna('NA').value_counts().rename_axis('source').reset_index(name='n')
print('\n=== Distribución por fuente ===')
print(source_dist.to_string(index=False))

# 5) Top unigramas y bigramas (modo temático, sin ruido funcional/editorial)
THEMATIC_NOISE_STOPWORDS = {
    'be', 'been', 'being', 'am', 'is', 'are', 'was', 'were',
    'do', 'does', 'did', 'done',
    'will', 'would', 'should', 'could', 'might', 'must', 'shall', 'can', 'may',
    'time', 'different', 'well', 'significant', 'important', 'potential', 'large',
    'present', 'observed', 'total', 'various', 'several', 'among', 'including',
    'based', 'according', 'results', 'result', 'show', 'shows', 'shown', 'suggest',
    'suggests', 'found', 'findings', 'approach', 'approaches', 'methodology',
    'analysis', 'analyses', 'data', 'new', 'high', 'low', 'future', 'past', 'current',
    'under', 'across', 'within', 'without',
    'long', 'term', 'series', 'carried', 'out',
    'open', 'access', 'creative', 'commons', 'competing', 'interests',
    'author', 'authors', 'key', 'words',
    'paper', 'study', 'studies', 'article', 'research',
}
ANALYSIS_STOPWORDS = STOPWORDS | THEMATIC_NOISE_STOPWORDS

counter_uni = Counter()
counter_bi = Counter()
for txt in eda_df['clean_abstract_lex'].fillna(''):
    toks = [fold_token(t) for t in txt.split()]
    toks = [t for t in toks if t and t not in ANALYSIS_STOPWORDS]
    counter_uni.update(toks)
    for i in range(len(toks) - 1):
        counter_bi[' '.join(toks[i:i+2])] += 1

top_uni = pd.DataFrame(counter_uni.most_common(20), columns=['unigrama', 'freq'])
top_bi = pd.DataFrame(counter_bi.most_common(20), columns=['bigrama', 'freq'])
print('\n=== Top 20 unigramas ===')
print(top_uni.to_string(index=False))
print('\n=== Top 20 bigramas ===')
print(top_bi.to_string(index=False))

# 6) Sesgos/huecos por descarte
dropped = trace_df[trace_df['filter_status'] == 'dropped'].copy()
reason_combo = dropped['filter_reason'].fillna('').value_counts().rename_axis('reason_combo').reset_index(name='n')
print('\n=== Top razones de descarte ===')
print(reason_combo.head(15).to_string(index=False))

## Gráficas para el informe (Tarea 5)

En esta sección generamos las visualizaciones clave de la rúbrica:

- publicaciones por año,
- distribución de longitud de abstracts,
- tabla de nulos como gráfico de barras.

La celda también guarda las figuras en `docs/eda/` para incluirlas directamente en el PDF.

In [ ]:
# Celda 9: Gráficas EDA y guardado a disco
# Qué hace: dibuja y guarda las gráficas pedidas para el informe.
# Qué devuelve: figuras en pantalla + rutas de archivo guardadas.

# Asegurar estructura de tablas por si se ejecuta esta celda de forma aislada
year_dist = (
    pd.to_numeric(eda_df['year'], errors='coerce')
    .dropna()
    .astype(int)
    .value_counts()
    .sort_index()
    .rename_axis('year')
    .reset_index(name='n')
)

null_table = pd.DataFrame({
    'campo': ['year', 'doi', 'journal', 'keywords', 'authors', 'abstract_norm'],
    'null_pct': [
        eda_df['year'].isna().mean() * 100,
        eda_df['doi'].isna().mean() * 100,
        eda_df['journal'].isna().mean() * 100,
        eda_df['keywords'].isna().mean() * 100,
        eda_df['authors'].isna().mean() * 100,
        (eda_df['abstract_norm'].str.len() == 0).mean() * 100,
    ],
})

fig1, ax1 = plt.subplots(figsize=(10, 4))
ax1.plot(year_dist['year'], year_dist['n'], marker='o', linewidth=1)
ax1.set_title('Publicaciones por año (corpus final)')
ax1.set_xlabel('Año')
ax1.set_ylabel('Nº de abstracts')
fig1.tight_layout()
path1 = EDA_DIR / 'publicaciones_por_anio.png'
fig1.savefig(path1, dpi=150)
plt.show()

fig2, ax2 = plt.subplots(figsize=(8, 4))
ax2.hist(eda_df['abstract_norm_len'], bins=30, edgecolor='black', alpha=0.85)
ax2.set_title('Distribución de longitud de abstracts normalizados')
ax2.set_xlabel('Longitud (caracteres)')
ax2.set_ylabel('Frecuencia')
fig2.tight_layout()
path2 = EDA_DIR / 'distribucion_longitud_abstracts.png'
fig2.savefig(path2, dpi=150)
plt.show()

fig3, ax3 = plt.subplots(figsize=(8, 4))
ax3.bar(null_table['campo'], null_table['null_pct'])
ax3.set_title('Porcentaje de nulos por campo')
ax3.set_xlabel('Campo')
ax3.set_ylabel('% nulos')
ax3.tick_params(axis='x', rotation=25)
fig3.tight_layout()
path3 = EDA_DIR / 'tabla_nulos_barras.png'
fig3.savefig(path3, dpi=150)
plt.show()

print('Gráficas guardadas en:')
print('-', path1)
print('-', path2)
print('-', path3)

## Exportación de resultados para el PDF

La siguiente celda deja todo persistido en disco para el entregable:

- `data/corpus/master_corpus_mixto_1000_clean_enriched.csv` (incluye `abstract_raw`, `abstract_norm`, `clean_abstract_lex`),
- tablas EDA en `docs/eda/*.csv`,
- resumen interpretativo en `docs/eda/eda_summary.md`.

In [ ]:
# Celda 11: Exportación de CSV y resumen EDA
# Qué hace: guarda dataset enriquecido y tablas derivadas para el reporte.
# Qué devuelve: rutas de salida y mini resumen interpretativo.

enriched_path = BASE_DIR / 'data/corpus/master_corpus_mixto_1000_clean_enriched.csv'
eda_df.to_csv(enriched_path, index=False)

year_dist.to_csv(EDA_DIR / 'year_distribution.csv', index=False)
null_table.to_csv(EDA_DIR / 'null_table.csv', index=False)
source_dist.to_csv(EDA_DIR / 'source_distribution.csv', index=False)
top_uni.to_csv(EDA_DIR / 'top_unigrams.csv', index=False)
top_bi.to_csv(EDA_DIR / 'top_bigrams.csv', index=False)
reason_combo.to_csv(EDA_DIR / 'drop_reason_combinations.csv', index=False)

summary_lines = [
    '# EDA del corpus limpio',
    '',
    f'- Documentos procesados: {n_processed}',
    f'- Con abstract no vacío (trazabilidad): {n_with_abstract}',
    f'- Final tras filtros: {n_final}',
    '',
    '## Longitud de abstracts normalizados',
    f"- Media: {eda_df['abstract_norm_len'].mean():.2f}",
    f"- Mediana: {eda_df['abstract_norm_len'].median():.2f}",
    f"- P10: {eda_df['abstract_norm_len'].quantile(0.10):.2f}",
    f"- P90: {eda_df['abstract_norm_len'].quantile(0.90):.2f}",
    '',
    '## Interpretación breve',
    '- Se observa pérdida relevante por filtro de longitud del abstract (<500).',
    '- DOI y keywords son los campos con más nulos en el corpus final.',
    '- La cobertura temporal incluye años antiguos y recientes; conviene contextualizar en el informe.',
]
summary_path = EDA_DIR / 'eda_summary.md'
summary_path.write_text('\n'.join(summary_lines), encoding='utf-8')

print('Archivos generados:')
print('-', enriched_path)
print('-', EDA_DIR / 'year_distribution.csv')
print('-', EDA_DIR / 'null_table.csv')
print('-', EDA_DIR / 'source_distribution.csv')
print('-', EDA_DIR / 'top_unigrams.csv')
print('-', EDA_DIR / 'top_bigrams.csv')
print('-', EDA_DIR / 'drop_reason_combinations.csv')
print('-', summary_path)

## Patrones de palabras y keywords + detección de años anómalos

Objetivo de esta sección:

- detectar papers con año > 2026 (posibles errores de metadatos),
- encontrar patrones en palabras frecuentes del abstract limpio,
- encontrar patrones en `keywords`,
- generar mapas de palabras globales y por grupos.

Sobre "cómo hacer grupos": en este notebook dejamos tres opciones prácticas:

1. Por frontera/carpeta temática (`pb_folder`) desde trazabilidad.
2. Por periodos temporales (`year_bin`).
3. Por fuente (`source`) o idioma (`language`).

In [ ]:
# Celda 13: Papers con año > 2026 (anomalías)
# Qué hace: lista los documentos con año futuro para revisión manual.
# Qué devuelve: tabla de papers posiblemente mal extraídos.

work_df = final_df.copy()
trace_kept = trace_df[trace_df['filter_status'] == 'kept'][['doc_id', 'pb_folder', 'source_folder']].drop_duplicates('doc_id')
work_df = work_df.merge(trace_kept, on='doc_id', how='left')

work_df['year_num'] = pd.to_numeric(work_df['year'], errors='coerce')
anomalias_year = work_df[work_df['year_num'] > 2026][
    ['doc_id', 'year', 'title', 'doi', 'source', 'pb_folder', 'source_folder']
].sort_values('year')

print('Total papers con año > 2026:', len(anomalias_year))
if len(anomalias_year) > 0:
    pd.set_option('display.max_colwidth', 120)
    print(anomalias_year.to_string(index=False))
else:
    print('No se detectaron años > 2026 en el corpus final.')

In [ ]:
# Celda 14: Patrones de palabras y keywords (global)
# Qué hace: calcula top palabras del abstract limpio y top keywords declaradas.
# Qué devuelve: tablas de frecuencia para análisis temático.

from collections import Counter

# Asegurar columnas limpias si no existen en memoria
if 'abstract_norm' not in work_df.columns:
    work_df['abstract_raw'] = work_df['abstract'].fillna('')
    work_df['abstract_norm'] = work_df['abstract_raw'].map(normalize_text)
    work_df['clean_abstract_lex'] = work_df['abstract_norm'].map(lex_clean)

# Filtro extra de seguridad al contar, para evitar términos funcionales remanentes.
EXTRA_STOP = {
    'are', 'was', 'were', 'been', 'being', 'be', 'am', 'is', 'use', 'used', 'using',
    'paper', 'study', 'article', 'abstract', 'introduction', 'keywords', 'results',
    'new', 'high', 'low', 'one', 'two', 'three', 'four', 'five', 'both', 'over'
}
ANALYSIS_STOPWORDS = STOPWORDS | EXTRA_STOP

# Top palabras (unigramas) sobre clean_abstract_lex
tokens = []
for txt in work_df['clean_abstract_lex'].fillna(''):
    cleaned = [fold_token(t) for t in txt.split()]
    cleaned = [t for t in cleaned if t and t not in ANALYSIS_STOPWORDS]
    tokens.extend(cleaned)
word_counts = Counter(tokens)
top_words = pd.DataFrame(word_counts.most_common(30), columns=['word', 'freq'])

# Top keywords declaradas por autores (separador principal ';')
kw_tokens = []
for kw in work_df['keywords'].fillna(''):
    parts = [fold_token(p.strip().lower()) for p in re.split(r'[;|,]', str(kw)) if p.strip()]
    parts = [p for p in parts if p and p not in ANALYSIS_STOPWORDS and len(p) >= 3]
    kw_tokens.extend(parts)
kw_counts = Counter(kw_tokens)
top_keywords = pd.DataFrame(kw_counts.most_common(30), columns=['keyword', 'freq'])

print('=== Top 30 palabras más repetidas (abstract limpio) ===')
print(top_words.to_string(index=False))
print('\n=== Top 30 keywords declaradas ===')
print(top_keywords.to_string(index=False))

In [ ]:
# Celda 15: Mapas de palabras globales (word clouds)
# Qué hace: genera mapa global de palabras de abstract y mapa global de keywords.
# Qué devuelve: dos figuras tipo "mapa de palabras".

from wordcloud import WordCloud

wc_words = WordCloud(width=1200, height=600, background_color='white', colormap='viridis')
wc_words.generate_from_frequencies(dict(top_words.values))

plt.figure(figsize=(12, 5))
plt.imshow(wc_words, interpolation='bilinear')
plt.axis('off')
plt.title('Mapa de palabras global - abstracts')
plt.show()

wc_kw = WordCloud(width=1200, height=600, background_color='white', colormap='plasma')
wc_kw.generate_from_frequencies(dict(top_keywords.values))

plt.figure(figsize=(12, 5))
plt.imshow(wc_kw, interpolation='bilinear')
plt.axis('off')
plt.title('Mapa de palabras global - keywords')
plt.show()

In [ ]:
# Celda 16: Patrones y mapas por grupos (configurable)
# Qué hace: agrupa documentos y muestra top términos + mapas por grupo.
# Qué devuelve: tablas por grupo y word clouds por cada grupo principal.

# 1) Crear bins temporales para agrupar por periodos
work_df['year_bin'] = pd.cut(
    work_df['year_num'],
    bins=[1899, 1999, 2009, 2014, 2019, 2024, 2026, 2100],
    labels=['<=1999', '2000-2009', '2010-2014', '2015-2019', '2020-2024', '2025-2026', '>2026']
)

# 2) Elige la variable de grupo aquí:
# Opciones recomendadas: 'pb_folder', 'year_bin', 'source', 'language'
GROUP_COL = 'pb_folder'

# Fallback si pb_folder no está disponible
if GROUP_COL == 'pb_folder' and work_df['pb_folder'].isna().all():
    GROUP_COL = 'source'

EXTRA_STOP = {
    'are', 'was', 'were', 'been', 'being', 'be', 'am', 'is', 'use', 'used', 'using',
    'paper', 'study', 'article', 'abstract', 'introduction', 'keywords', 'results',
    'new', 'high', 'low', 'one', 'two', 'three', 'four', 'five', 'both', 'over'
}
ANALYSIS_STOPWORDS = STOPWORDS | EXTRA_STOP

print('Agrupando por:', GROUP_COL)

# 3) Selección de grupos con suficiente tamaño
group_sizes = work_df[GROUP_COL].fillna('NA').value_counts()
selected_groups = group_sizes.head(4).index.tolist()  # top 4 grupos más grandes
print('\nTamaño por grupo (top 10):')
print(group_sizes.head(10).to_string())

for grp in selected_groups:
    sub = work_df[work_df[GROUP_COL].fillna('NA') == grp]

    # Top palabras de abstract limpio
    toks = []
    for txt in sub['clean_abstract_lex'].fillna(''):
        cleaned = [fold_token(t) for t in txt.split()]
        cleaned = [t for t in cleaned if t and t not in ANALYSIS_STOPWORDS]
        toks.extend(cleaned)
    cnt = Counter(toks)
    top_grp_words = pd.DataFrame(cnt.most_common(15), columns=['word', 'freq'])

    # Top keywords declaradas
    kw_toks = []
    for kw in sub['keywords'].fillna(''):
        parts = [fold_token(p.strip().lower()) for p in re.split(r'[;|,]', str(kw)) if p.strip()]
        parts = [p for p in parts if p and p not in ANALYSIS_STOPWORDS and len(p) >= 3]
        kw_toks.extend(parts)
    kw_cnt = Counter(kw_toks)
    top_grp_kw = pd.DataFrame(kw_cnt.most_common(15), columns=['keyword', 'freq'])

    print('\n' + '=' * 80)
    print(f'Grupo: {grp} | n={len(sub)}')
    print('Top palabras:')
    print(top_grp_words.to_string(index=False))
    print('Top keywords:')
    print(top_grp_kw.to_string(index=False))

    # Word cloud palabras
    if len(top_grp_words) > 0:
        wc_g_words = WordCloud(width=900, height=420, background_color='white', colormap='viridis')
        wc_g_words.generate_from_frequencies(dict(top_grp_words.values))
        plt.figure(figsize=(10, 4))
        plt.imshow(wc_g_words, interpolation='bilinear')
        plt.axis('off')
        plt.title(f'Mapa de palabras (abstract) - {GROUP_COL}: {grp}')
        plt.show()

    # Word cloud keywords
    if len(top_grp_kw) > 0:
        wc_g_kw = WordCloud(width=900, height=420, background_color='white', colormap='magma')
        wc_g_kw.generate_from_frequencies(dict(top_grp_kw.values))
        plt.figure(figsize=(10, 4))
        plt.imshow(wc_g_kw, interpolation='bilinear')
        plt.axis('off')
        plt.title(f'Mapa de palabras (keywords) - {GROUP_COL}: {grp}')
        plt.show()

## Agrupación temática automática (sin PB, sin TF-IDF)

En esta sección creamos grupos temáticos **solo** por contenido textual (abstract limpio + keywords), sin usar `pb_folder` ni TF-IDF.

Técnica:

- Matriz de conteo (`CountVectorizer`) con unigramas y bigramas.
- Reducción semántica con LSA (`TruncatedSVD`) para capturar coocurrencias latentes.
- Clustering aglomerativo con distancia coseno sobre el espacio semántico.
- Selección automática de `k` por *silhouette score* (coseno).
- Etiqueta interpretable por grupo usando palabras frecuentes de cada cluster.

In [ ]:
# Celda 18: Clustering semántico de papers (sin pb_folder, sin TF-IDF)
# Qué hace: agrupa papers por espacio semántico LSA sobre abstract+keywords.
# Qué devuelve: k óptimo, tabla de grupos, etiquetas temáticas y ejemplos.

import numpy as np
from collections import Counter
from sklearn.cluster import AgglomerativeClustering
from sklearn.decomposition import TruncatedSVD
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.metrics import silhouette_score
from sklearn.preprocessing import Normalizer

# Fallback por si no se ejecutó antes la celda EDA que define ANALYSIS_STOPWORDS
if 'ANALYSIS_STOPWORDS' not in globals():
    ANALYSIS_STOPWORDS = STOPWORDS

# 1) Texto para clustering (sin tocar abstract original)
topic_df = eda_df.copy()
topic_df['kw_norm'] = (
    topic_df['keywords']
    .fillna('')
    .astype(str)
    .str.lower()
    .str.replace(r'[;|,]+', ' ', regex=True)
)

CLUSTER_NOISE = {
    'university', 'department', 'institute', 'correspondence', 'author', 'authors',
    'published', 'review', 'article', 'copyright', 'license', 'attribution',
    'frontiers', 'open', 'access', 'received', 'accepted', 'editor', 'email',
}

raw_topic_text = (
    topic_df['clean_abstract_lex'].fillna('').astype(str) + ' ' + topic_df['kw_norm']
).str.replace(r'\s+', ' ', regex=True).str.strip()

def topic_clean(text: str) -> str:
    toks = [fold_token(t) for t in re.findall(r"[a-zA-ZÀ-ÿ]{3,}", str(text).lower())]
    toks = [t for t in toks if t and t not in ANALYSIS_STOPWORDS and t not in CLUSTER_NOISE]
    return ' '.join(toks)

topic_df['text_for_topic'] = raw_topic_text.map(topic_clean)
topic_df = topic_df[topic_df['text_for_topic'].str.len() > 0].copy()

# 2) Espacio semántico LSA (Count -> SVD -> Normalize)
vectorizer = CountVectorizer(
    ngram_range=(1, 2),
    min_df=5,
    max_df=0.85,
)
X_counts = vectorizer.fit_transform(topic_df['text_for_topic'])

n_components = min(100, max(20, X_counts.shape[1] - 1))
svd = TruncatedSVD(n_components=n_components, random_state=42)
X_lsa = svd.fit_transform(X_counts)
X = Normalizer(copy=False).fit_transform(X_lsa)

# 3) Selección de k por silhouette (coseno) + estabilidad de tamaño de cluster
def make_agg_model(k: int):
    try:
        return AgglomerativeClustering(n_clusters=k, metric='cosine', linkage='average')
    except TypeError:
        return AgglomerativeClustering(n_clusters=k, affinity='cosine', linkage='average')

k_candidates = list(range(3, 9))
MIN_CLUSTER_PCT = 0.03  # evita grupos minúsculos (<3% del corpus)
scores = []

for k in k_candidates:
    model_k = make_agg_model(k)
    labels_k = model_k.fit_predict(X)
    score_k = silhouette_score(X, labels_k, metric='cosine')
    counts_k = pd.Series(labels_k).value_counts(normalize=True)
    min_cluster_pct = float(counts_k.min())
    is_valid = min_cluster_pct >= MIN_CLUSTER_PCT
    scores.append({
        'k': k,
        'silhouette_cosine': score_k,
        'min_cluster_pct': min_cluster_pct,
        'valid_size': is_valid,
    })

scores_df = pd.DataFrame(scores).sort_values('silhouette_cosine', ascending=False)
valid_scores = scores_df[scores_df['valid_size']]
if not valid_scores.empty:
    best_k = int(valid_scores.iloc[0]['k'])
else:
    best_k = int(scores_df.iloc[0]['k'])

print('=== Selección de k (silhouette coseno + tamaño mínimo de cluster) ===')
print(scores_df.to_string(index=False, float_format=lambda x: f'{x:.4f}'))
print(f'\nK óptimo seleccionado: {best_k}')

# 4) Clustering final
cluster_model = make_agg_model(best_k)
labels = cluster_model.fit_predict(X)
topic_df['cluster_id'] = labels

# 5) Etiquetas interpretables por cluster (palabras frecuentes)
cluster_labels = {}
for c in sorted(topic_df['cluster_id'].unique()):
    docs = topic_df.loc[topic_df['cluster_id'] == c, 'text_for_topic']
    toks = []
    for doc in docs:
        toks.extend(doc.split())
    top_terms = [w for w, _ in Counter(toks).most_common(8)]
    cluster_labels[c] = ', '.join(top_terms[:4]) if top_terms else 'sin_terminos'

topic_df['tema_auto'] = topic_df['cluster_id'].map(lambda c: f'Tema {c}: {cluster_labels[c]}')

# 6) Resumen de grupos
cluster_summary = (
    topic_df.groupby(['cluster_id', 'tema_auto'])
    .size()
    .reset_index(name='n_docs')
    .sort_values('n_docs', ascending=False)
)
cluster_summary['pct_docs'] = (cluster_summary['n_docs'] / len(topic_df) * 100).round(2)

print('\n=== Resumen de grupos automáticos ===')
print(cluster_summary.to_string(index=False))

# 7) Ejemplos por grupo (3 títulos)
print('\n=== Ejemplos por grupo (3 títulos) ===')
for _, row in cluster_summary.iterrows():
    cid = int(row['cluster_id'])
    label = row['tema_auto']
    sample_titles = topic_df.loc[topic_df['cluster_id'] == cid, 'title'].dropna().head(3).tolist()
    print('\n' + '-' * 90)
    print(f'{label} | n={int(row["n_docs"])} ({row["pct_docs"]}%)')
    for i, t in enumerate(sample_titles, 1):
        print(f'{i}. {t}')

# 8) Guardar resultados para informe
cluster_out = topic_df[[
    'doc_id', 'title', 'year', 'doi', 'source', 'keywords', 'cluster_id', 'tema_auto'
]].copy()
cluster_out.to_csv(EDA_DIR / 'semantic_topic_clusters.csv', index=False)
cluster_summary.to_csv(EDA_DIR / 'semantic_topic_cluster_summary.csv', index=False)

print('\nCSV guardados:')
print(EDA_DIR / 'semantic_topic_clusters.csv')
print(EDA_DIR / 'semantic_topic_cluster_summary.csv')

## Control de calidad de términos por documento (CSV final)

Objetivo:

- comprobar si `top_terms_no_stopwords` en el CSV final contiene stopwords indeseadas,
- auditar de forma explícita palabras como `are`, `was`, `were`, `been`, etc.,
- dejar trazabilidad antes de tocar archivos finales.

In [ ]:
# Celda: Auditoría de stopwords en top_terms_no_stopwords (por documento)
# Qué hace: revisa el CSV final y detecta documentos con términos prohibidos en top_terms_no_stopwords.
# Qué devuelve: conteo de documentos afectados + ejemplos para revisión manual.

qc_df = pd.read_csv(FINAL_CSV)

basic_stop = {
    'are', 'was', 'were', 'been', 'being', 'be', 'am', 'is',
    'the', 'and', 'for', 'with', 'this', 'that', 'from'
}

rows_with_stop = []
for _, r in qc_df[['doc_id', 'top_terms_no_stopwords']].fillna('').iterrows():
    toks = [fold_token(t.strip().lower()) for t in str(r['top_terms_no_stopwords']).split(';') if t.strip()]
    hits = [t for t in toks if t in basic_stop]
    if hits:
        rows_with_stop.append({
            'doc_id': r['doc_id'],
            'hits': '; '.join(sorted(set(hits))),
            'top_terms': '; '.join(toks[:12]),
        })

qc_hits = pd.DataFrame(rows_with_stop)
print('Total documentos en CSV final:', len(qc_df))
print('Documentos con stopwords en top_terms_no_stopwords:', len(qc_hits))

if len(qc_hits) > 0:
    print('\nEjemplos con problema:')
    print(qc_hits.head(15).to_string(index=False))
else:
    print('\nOK: no se detectaron stopwords básicas en top_terms_no_stopwords.')

## Bitacora de ejecucion y analisis de outputs (hoy)

Esta seccion documenta, celda por celda, que se ejecuto, que devolvio y que conclusion operativa se extrae.

### Celdas 1 a 3: Carga y preparacion

- Celda 2: Carga `data/corpus/master_corpus_mixto_1000_clean.csv` y `data/corpus/master_corpus_mixto_1000_traceability.csv`.
  - Output clave: corpus final = **700** registros, trazabilidad = **1000** registros.
  - Conclusion: el pipeline mantiene 700 documentos validos y conserva auditoria completa de 1000.

- Celda 3: Define funciones de normalizacion y stopwords de trabajo.
  - Output clave: confirmacion de filas y columnas del corpus final.
  - Conclusion: la base de limpieza y EDA esta activa y reutilizable para todas las celdas siguientes.

### Celdas 4 y 5: Limpieza textual para analisis

- Celda 5: Crea `abstract_raw`, `abstract_norm`, `clean_abstract_lex`, `abstract_norm_len`.
  - Output clave: preview comparativo antes/despues y estadistica de longitudes.
  - Conclusion: el abstract original se conserva y la limpieza se aplica en columnas derivadas.

### Celdas 6 y 7: EDA tabular

- Celda 7: Cobertura, nulos, distribucion temporal, top terminos y razones de descarte.
  - Output clave:
    - Documentos procesados: **1000**
    - Final tras filtros: **700**
    - Descartados: **300**
  - Conclusion: la perdida principal se explica por filtros de calidad y no por fallo de ejecucion.

### Celdas 8 y 9: Graficas

- Celda 9: Genera y guarda figuras (`publicaciones_por_anio.png`, `distribucion_longitud_abstracts.png`, `tabla_nulos_barras.png`).
  - Output clave: rutas de guardado confirmadas.
  - Conclusion: artefactos visuales listos para informe.

### Celdas 10 y 11: Exportacion

- Celda 11: Exporta dataset enriquecido y tablas EDA.
  - Output clave: se guardan `top_unigrams.csv`, `top_bigrams.csv`, `drop_reason_combinations.csv` y resumen markdown.
  - Conclusion: el estado analitico queda persistido fuera del notebook.

### Celdas 12 a 16: Patrones y mapas

- Celda 13: Deteccion de anomalias de ano.
  - Output clave: papers con `year > 2026` = **7**.
  - Conclusion: hay casos para revision de metadatos temporales.

- Celda 14: Top global de palabras y keywords.
  - Output clave actual (top 10 unigramas):
    - `climate, change, water, global, land, changes, surface, species, environmental, system`
  - Conclusion: el top es tematico y ya no esta dominado por stopwords basicas.

- Celda 15: Wordcloud global.
  - Output clave: mapas de palabras para abstracts y keywords.
  - Conclusion: coherencia visual con el ranking textual.

- Celda 16: Mapas por grupo configurable.
  - Output clave: comparativa por grupos con top palabras y top keywords.
  - Conclusion: permite leer diferencias tematicas entre subcorpus.

### Celdas 17 a 20: Agrupacion tematica automatica (sin PB)

- Celda 20: Clustering semantico con Count + LSA + coseno.
  - Output clave de hoy:
    - `k` elegido por silhouette: **6**
    - Reparto principal:
      - Tema 4 (aerosol/ocean/ozone/surface): **214 (30.57%)**
      - Tema 0 (research/environmental/systems/change): **206 (29.43%)**
      - Tema 2 (climate/change/land/water): **185 (26.43%)**
      - Tema 1 (species/marine/ecosystem): **87 (12.43%)**
      - Temas 3 y 5: **4 docs** cada uno (microgrupos)
  - Conclusion: la estructura macrotematica es valida, pero hay microgrupos outlier que conviene fusionar o filtrar.

### Celdas 22 y 23: Control de calidad del CSV final

- Celda 23: Auditoria explicita de stopwords en `top_terms_no_stopwords` (por documento).
  - Output clave: documentos con stopwords basicas (`are`, `was`, `were`, `been`, etc.) = **0**.
  - Conclusion: los terminos por documento del CSV final estan limpios de stopwords basicas.

## Conclusiones sanas y accionables

1. El corpus final y la trazabilidad estan consistentes (700/1000) y auditables.
2. Los terminos mas repetidos actuales son tematicos; no aparecen stopwords basicas en `top_terms_no_stopwords`.
3. El analisis semantico sin PB funciona para macrotemas, pero produce 2 microgrupos outlier; recomendacion: fusionar clusters <2% o fijar `k=4` para reporte final.
4. El abstract original se conserva; la limpieza solo impacta columnas derivadas para analisis.
5. El estado de salida en CSV y figuras es apto para documentacion de entrega.